# 01. 환경 세팅 및 생성/분리 단독 테스트

이 노트북은 Colab 무료 T4 GPU에서 실행하는 것을 전제로 한다.

실행 순서: 런타임 -> 런타임 유형 변경 -> T4 GPU 선택 후 아래 셀을 순서대로 실행한다.

목적: MusicGen 단독 생성, Demucs 단독 분리가 각각 정상 동작하는지 확인한다. 재생성/재조합 로직은 이후 노트북에서 다룬다.

## 0. GPU 확인

In [ ]:
!nvidia-smi

## 1. 라이브러리 설치

AudioCraft(MusicGen)와 Demucs를 설치한다. Colab은 Linux 기반이라 별도 WSL 없이 바로 설치된다.

In [ ]:
!python3 -m pip install -q -U git+https://github.com/facebookresearch/audiocraft
!python3 -m pip install -q -U demucs
!apt-get -qq install -y ffmpeg

## 2. Google Drive 마운트 (중간 산출물 저장용)

Colab 무료 티어는 세션이 끊기면 로컬 파일이 사라진다. 생성/분리 결과를 Drive에 저장해 재개 가능하도록 한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/stem-remix-assistant/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(OUTPUT_DIR)

## 3. MusicGen 단독 생성 테스트

medium 모델, stereo, 짧은 길이(15초)로 우선 테스트한다. 문제 없으면 길이를 늘린다.

In [ ]:
import torch
from audiocraft.models import MusicGen
from audiocraft.data.audio import audio_write

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

model = MusicGen.get_pretrained('facebook/musicgen-stereo-medium', device=device)
model.set_generation_params(duration=15)

prompt = ["lofi hiphop with mellow piano, 90 BPM, warm and relaxed"]
wav = model.generate(prompt)

for idx, one_wav in enumerate(wav):
    audio_write(f'{OUTPUT_DIR}/draft_{idx}', one_wav.cpu(), model.sample_rate, strategy="loudness")

print('생성 완료')

## 4. Demucs 단독 분리 테스트

방금 생성한 트랙을 4-스템(드럼/베이스/보컬/기타)으로 분리한다.

In [ ]:
!demucs -n htdemucs "{OUTPUT_DIR}/draft_0.wav" -o "{OUTPUT_DIR}/separated" 

## 5. 결과 확인

In [ ]:
import IPython.display as ipd
import glob

for f in sorted(glob.glob(f'{OUTPUT_DIR}/separated/htdemucs/draft_0/*.wav')):
    print(f)
    display(ipd.Audio(f))

## 6. 체크리스트

- [ ] GPU 정상 인식
- [ ] MusicGen 생성 성공 (에러 없이 wav 생성)
- [ ] Demucs 분리 성공 (4개 스템 파일 생성)
- [ ] Drive에 결과 저장 확인

문제가 생기면 `docs/troubleshooting/`에 날짜별로 기록한다.